In [0]:
%pip install fitparse

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import savgol_filter
import os
import glob

from utils.triathlon_utils import (
    parse_fit, 
    clean_fit_data, 
    calculate_elapsed_time, 
    apply_triathlon_style,
    format_time_axis,
    export_for_instagram,
    KLEUREN_PALETTE
)

from utils.config import DEFAULT_ATHLETE, DEFAULT_PATHS

In [0]:
# Geautomatiseerde batch-analyse 
def run_stubaier_baseline_batch(fit_dir):
    """
    Leest alle FIT-bestanden op basis van de exacte kolomnamen ('distance', 'enhanced_altitude', 'cadence'),
    berekent de klim-cadans (>5% gradient) en filtert op de baseline van 12 t/m 18 sept.
    """
    search_path = os.path.join(fit_dir, "*.fit")
    fit_files = sorted(glob.glob(search_path))
    
    if not fit_files:
        print(f"Geen FIT-bestanden gevonden in: {fit_dir}")
        return None, None
        
    results = []
    print("Start automatische analyse en baseline selectie...\n" + "-"*50)
    
    for file_path in fit_files:
        filename = os.path.basename(file_path)
        date_prefix = filename.split('_')[0]
        
        try:
            # 1. Parse en opschonen
            df_raw = parse_fit(file_path)
            df_stage = clean_fit_data(df_raw)
            
            # 2. Controleer of de juiste kolommen aanwezig zijn
            if 'distance' in df_stage.columns and 'enhanced_altitude' in df_stage.columns and 'cadence' in df_stage.columns:
                df_work = df_stage.copy()
                
                # Zet nullen (stilstaan) om naar NaN
                df_work['cadence'] = df_work['cadence'].replace(0, np.nan)
                
                # Afstand in kilometers omzetten voor de gradient berekening
                df_work['dist_km'] = df_work['distance'] / 1000.0
                
                # Gradient (%) berekenen: delta hoogte / delta afstand (in meters)
                delta_alt = df_work['enhanced_altitude'].diff()
                delta_dist = df_work['distance'].diff() # al in meters
                delta_dist = delta_dist.replace(0, np.nan)
                df_work['gradient_pct'] = (delta_alt / delta_dist) * 100.0
                
                # Filter klimmen (> 5%)
                df_climbs = df_work[df_work['gradient_pct'] >= 5.0]
                avg_climb_cadence = df_climbs['cadence'].mean()
                
                # Bepaal of dit binnen het baseline-bereik valt (12 t/m 18 september)
                is_baseline_period = "20260912" <= date_prefix <= "20260918"
                
                results.append({
                    'filename': filename,
                    'date': date_prefix,
                    'avg_climb_cadence': avg_climb_cadence,
                    'used_for_baseline': is_baseline_period
                })
                
                tag = "🎯 [BASELINE WEEK 1]" if is_baseline_period else "📈 [LATERE ETAPPE]"
                print(f"{tag} {filename} -> Klim-cadans (>5%): {avg_climb_cadence:.1f} rpm")
            else:
                print(f"⚠ Overgeslagen (missende kolommen): {filename}")
                
        except Exception as e:
            print(f"❌ Fout bij verwerken van {filename}: {e}")
            
    df_summary = pd.DataFrame(results)
    
    if not df_summary.empty:
        # Filter alleen de dagen van 12 t/m 18 september voor de baseline
        baseline_df = df_summary[df_summary['used_for_baseline'] == True]
        baseline_week1 = baseline_df['avg_climb_cadence'].mean()
        
        print("-" * 50)
        print(f"✅ Baseline berekend over {len(baseline_df)} etappes (12-18 sept): {baseline_week1:.1f} rpm")
        print("-" * 50)
        
        return df_summary, baseline_week1
        
    return None, None

# Uitvoeren in Databricks
summary_df, baseline_val = run_stubaier_baseline_batch(DEFAULT_PATHS.fit_dir)

In [0]:
def create_terrain_filtered_cadence_chart(
    df_stage,
    terrain_type='climb',
    baseline_climb_cadence=51.8,
    stage_num=8,
    window_length=51,
    polyorder=3,
    elev_col='enhanced_altitude'
):
    """Genereert een strakke, gelijkmatige cadans-grafiek door interpolatie op een 
    uniform rooster, met een neutrale grijze achtergrond voor het hoogteprofiel.
    """
    plt.style.use('dark_background')
    fig, ax1 = plt.subplots(figsize=(12, 6), facecolor='black')
    ax1.set_facecolor('black')
    
    COLOR_CYAN = '#00F5FF'      # Cyaan/Blauw voor de cadanslijn
    COLOR_BASELINE = '#FF2A6D'  # Roze/Magenta voor de baseline
    COLOR_ELEVATION = '#4B5563' # Neutraal grijs voor het hoogteprofiel
    
    # Bepaal de juiste hoogtekolom (val terug op 'elevation' indien 'enhanced_altitude' ontbreekt)
    alt_col = elev_col if elev_col in df_stage.columns else 'elevation'
    
    if 'distance' in df_stage.columns and alt_col in df_stage.columns and 'cadence' in df_stage.columns:
        df_work = df_stage.copy()
        df_work['cadence'] = df_work['cadence'].replace(0, np.nan)
        df_work['dist_km'] = df_work['distance'] / 1000.0
        
        # 1. Maak een uniform afstandsraster (elke 10 meter) om gaten te voorkomen
        min_dist = df_work['dist_km'].min()
        max_dist = df_work['dist_km'].max()
        uniform_dist = np.arange(min_dist, max_dist, 0.01) # stappen van 10 meter
        
        # Interpoleer hoogte en cadans naar dit uniforme rooster
        interp_alt = np.interp(uniform_dist, df_work['dist_km'], df_work[alt_col].interpolate())
        interp_cad = np.interp(uniform_dist, df_work['dist_km'], df_work['cadence'].interpolate())
        
        df_uniform = pd.DataFrame({
            'dist_km': uniform_dist,
            alt_col: interp_alt,
            'cadence': interp_cad
        })
        
        # Bereken gradient op het uniforme rooster
        delta_alt = np.gradient(df_uniform[alt_col])
        delta_dist = np.gradient(df_uniform['dist_km']) * 1000.0 # terug naar meters
        df_uniform['gradient_pct'] = (delta_alt / delta_dist) * 100.0
        
        # 2. Filter op basis van gekozen terreintype
        if terrain_type == 'climb':
            df_filtered = df_uniform[df_uniform['gradient_pct'] >= 5.0].copy()
            title_label = f'ETAPPE PERFORMANCE: CADANSTREND BIJ KLIMMEN VAN +5.0%'
        elif terrain_type == 'descend':
            df_filtered = df_uniform[df_uniform['gradient_pct'] <= -5.0].copy()
            title_label = f'ETAPPE PERFORMANCE: CADANSTREND BIJ KLIMMEN VAN +5.0%'
        else:
            df_filtered = df_uniform[(df_uniform['gradient_pct'] >= -5.0) & (df_uniform['gradient_pct'] <= 5.0)].copy()
            title_label = f'ETAPPE PERFORMANCE: CADANSTREND BIJ KLIMMEN VAN +5.0%'
            
        x_vals = df_filtered['dist_km']
        
        # 3. Vloeiende cadans berekenen op de gefilterde data
        if len(df_filtered) > 15: # Zorg dat er genoeg datapunten zijn
            cadence_clean = df_filtered['cadence'].interpolate(method='linear').ffill().bfill()
            
            # Bepaal een dynamische vensterlengte op basis van de hoeveelheid data (moet oneven zijn)
            window_size = min(75, len(cadence_clean) // 2 * 2 + 1)
            if window_size < 5:
                window_size = 5
                
            # Pas de Savitzky-Golay filter toe (window_size, polyorder=3)
            cadence_smooth = savgol_filter(cadence_clean, window_length=window_size, polyorder=3)
        elif len(df_filtered) > 1:
            cadence_clean = df_filtered['cadence'].interpolate(method='linear').ffill().bfill()
            cadence_smooth = cadence_clean.rolling(window=min(15, len(cadence_clean)), min_periods=1).mean()
        else:
            cadence_smooth = df_filtered['cadence']
            
        # 4. Secundaire as voor het hoogteprofiel met strakke grijze look
        min_elev = df_uniform[alt_col].min()
        max_elev = df_uniform[alt_col].max()
        
        ax2 = ax1.twinx()
        ax2.set_facecolor('none')
        ax2.fill_between(df_uniform['dist_km'], df_uniform[alt_col], color=COLOR_ELEVATION, alpha=0.3, zorder=1)
        ax2.plot(df_uniform['dist_km'], df_uniform[alt_col], color='#9CA3AF', linewidth=1, alpha=0.6, zorder=2)
        ax2.set_ylabel('Hoogte (m)', color='#9CA3AF', fontsize=9)
        ax2.tick_params(colors='#9CA3AF', labelsize=8)
        ax2.grid(False)

        ax2.set_ylim(min_elev - 50, max_elev + (max_elev - min_elev) * 0.3)

        # 5. Plotten op de primaire as (cadans en baseline)
        ax1.plot(x_vals, cadence_smooth, color=COLOR_CYAN, lw=2.2, label=f'Etappe {stage_num}', zorder=4)
        ax1.axhline(y=baseline_climb_cadence, color=COLOR_BASELINE, linestyle='--', lw=1.8, label=f'Baseline week 1', zorder=3)
        
        if not df_filtered.empty and df_filtered['cadence'].notna().any():
            actual_avg = df_filtered['cadence'].mean()
        else:
            actual_avg = 0.0
            
        diff = actual_avg - baseline_climb_cadence
        
        stats_text = (
            f"Gem. Cadans Etappe {stage_num}: {actual_avg:.1f} rpm\n"
            f"Baseline Week 1: {baseline_climb_cadence:.1f} rpm\n"
            f"---------------------------\n"
            f"Verschil: {diff:+.1f} rpm"
        )
        
        props = dict(boxstyle='round,pad=0.5', facecolor='#111827', alpha=0.85, edgecolor='#374151')
        ax1.text(
            0.02, 0.95, stats_text, 
            transform=ax1.transAxes, 
            fontsize=10, color='white', fontweight='bold',
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='#111827', alpha=0.85, edgecolor='#374151')
        )

        

        ax1.set_xlabel('Afstand (km)', color='white', fontsize=10, fontweight='bold')
        ax1.set_ylabel('Cadans (rpm)', color=COLOR_CYAN, fontsize=10, fontweight='bold')
        ax1.set_title(title_label, color='white', fontsize=11, fontweight='bold', pad=12)
        ax1.set_zorder(ax2.get_zorder() + 1)
        ax1.set_frame_on(False)
    else:
        ax1.text(0.5, 0.5, 'Niet alle benodigde kolommen gevonden', color='white', ha='center', va='center')
        
    ax1.tick_params(colors='white', labelsize=9)
    ax1.grid(False)
    
    lines_1, labels_1 = ax1.get_legend_handles_labels()
    ax1.legend(lines_1, labels_1, loc='lower right', facecolor='#121212', edgecolor='#333333', labelcolor='white', fontsize=8)
    
    plt.tight_layout()
    return fig

In [0]:
# Het FIT-bestand van etappe 8
fit_file_path = f"{DEFAULT_PATHS.fit_dir}/20260919_stubaier_hohenweg_etappe_8.fit"

# 1. Inlezen & Opschonen via de utils
df = parse_fit(fit_file_path)
df_clean = clean_fit_data(df)

# 2. Grafiek genereren
fig_fatigue = create_terrain_filtered_cadence_chart(
    df_stage=df_clean,  
    baseline_climb_cadence=baseline_val, 
    stage_num=8
)

# 3. Exporteren voor Instagram 
export_for_instagram(fig_fatigue, filename="20260919_accumulated_fatigue_cadence", format_type='landscape')
